In [1]:
#from scipy import constants as const
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

import oreonspy as op
from oreonspy import utils as ut

import logging

import gc; gc.collect()

0

Generate a set of cavites with parameters spanning in the desired range. The total number of cavities will not exceed the numer_of_cavities. Each cavity will be saved in an XML file.

In [ ]:
import xml.etree.ElementTree as ET

# Parse the existing XML file
tree = ET.parse('../tests/optical_cavities.xml')
#root = tree.getroot()  # Commented to avoid unnecessary regeneration of cavities testset

# Iterate through each cavity entry in the XML file
for cavity_entry in root.findall('Cavity'):
    L = float(cavity_entry.find('Length').text)
    Finesse = float(cavity_entry.find('Finesse').text)
    UUID = cavity_entry.find('UUID').text

    # Calculate r_a and r_b using the utility function
    r_a_r_b_product = ut.r_a_r_b(Finesse)
    r_a = np.sqrt(r_a_r_b_product)
    r_b = np.sqrt(r_a_r_b_product)

    # Create a new cavity object
    cavity = op.Cavity(t_a=.014, r_a=r_a, r_b=r_b, L=L, debug=False)

    # Save the cavity to a new XML file
    filename = f"../tests/optical_cavities_testset/cavity_{UUID}.xml"
    cavity.xml_save(filename)
    print(f"Converted and saved cavity {UUID} to {filename}")

AttributeError: 'list' object has no attribute 'findall'

In [ ]:
# LASER
E_in_avg = 1.+0.j  #

def generate_cavity_evolution(cavity, z_evolution, E_evolution, title=None):
    results = np.zeros(num_points, dtype=np.complex128)

    for t in range(num_points):
        results[t], _ = cavity.sim_step(E_evolution[t], d_zeta=z_evolution[t]-z_evolution[t-1], )
    
    s = np.abs(results)**2
    ph = np.angle(results)
    pdh = ut.V_pdh(0., E_evolution, results)
    
    print(f"Results for v = {velocity_factor} v_cr and f_calc = {f_calc} Hz:")
    #print(results)

    simple_plot = False
    if simple_plot is False:
        ut.plot_cavity_evolution(z_evolution, E_evolution, s, ph, pdh, zeta1_positons=None, s_ref=None, ph_ref=None, title=title)
    else:
        plt.plot(time_steps, s)
        plt.plot(time_steps, ph)
        plt.plot(time_steps, pdh)
        plt.xlabel('Time (s)')
        plt.ylabel('Cavity Evolution')
        plt.title(f'Cavity Evolution for v = {velocity_factor} v_cr and f_calc = {f_calc} Hz')
        plt.show()

In [ ]:
import os
from IPython import get_ipython

# Define velocities and f_calc range
velocity_factors = np.array([0.1, 1, 2, 5, 10])
frequency_factors = np.array([1, 3, 10])

# Generate cavity evolution for each cavity in the XML file
#xml_files = ["TEst_params.xml"]
# Get all XML files from the specified directory
xml_files = tuple(
    os.path.join("../tests/optical_cavities_testset", file)
    for file in os.listdir("../tests/optical_cavities_testset")
    if file.endswith(".xml")
)

for xml_file in xml_files:
    f_path = os.path.splitext(xml_file)[0]

    cavity = op.Cavity(debug=logging.DEBUG, log_file=f_path+".log")
    cavity.xml_load(xml_file)

    v_cr = ut.critical_velocity(cavity, ut.lambd)
    
    counter = 0
    for velocity_factor in velocity_factors:
        velocity = velocity_factors * v_cr
        f_calc_range = frequency_factors * ut.optimal_sampling_frequency(cavity, velocity_factor)
        print(f"Freq: {ut.optimal_sampling_frequency(cavity, velocity_factor)}")
        for f_calc in f_calc_range:
            print(f"Calculating for v = {velocity_factor} and f_calc = {f_calc} Hz")
            num_points, time_steps = ut.generate_time_points_for_constant_velocity(ut.critical_velocity(cavity, ut.lambd)*velocity_factor, f_calc)
            
            E_ev = np.ones(num_points)

            cavity.simulation(ut.k, f_calc, E_ev[0])

            z_ev = np.linspace(0., ut.lambd, num_points)  # Only here cavity.Theta is known

            generate_cavity_evolution(cavity, z_ev, E_ev, title=f_path+"-"+str(counter))
            counter += 1

    get_ipython().history_manager.reset()
    del num_points, time_steps, cavity, E_ev, z_ev

    gc.collect()
    


In [ ]:
cavity = op.Cavity(t_a=1., r_a=0.97, r_b=0.999, L=3000, debug=False)

In [ ]:
print(cavity.tau())
print(cavity.tau_s())  #? 20 times larger

my_k = ut.k

def Omega(t, v, cavity):
    # 2.70 
    return -my_k*v*t/cavity.T

# parameter space candidates: Finesse, length


In [ ]:
cav_tau = cavity.tau()

t_end = cav_tau*1

t_ax = np.linspace(0,t_end,100)

oscillation_freq = Omega(t_ax, -1*ut.critical_velocity(cavity, ut.lambd), cavity)
oscillation_freq[0] = oscillation_freq[1]
print(oscillation_freq)

oscillation_period = 1/oscillation_freq
plt.plot(t_ax, oscillation_period, label="oscillation period")
plt.plot(t_ax, np.ones(len(t_ax))*cav_tau, label="tau")
plt.legend()
plt.show()

fmin_idx = 0.
for i, f in enumerate(oscillation_period):
    if f < cav_tau:
        fmin_idx = i
        break


t_start = t_ax[int(fmin_idx)]
t_NL_useful = t_end - t_start
print("Useful oscillation duration: {0}".format(t_NL_useful))

plt.plot(t_ax, cav_tau*oscillation_freq)
plt.show()

# Jeżeli przecięcie oscillation period i tau następuje wewnątrz wykresu (wykres kończy się na tau, sprawdzić czy nie powinen kończyć się później) to mamy do czynienia z przejściem oscylującym.
# Tau zależy od Finesse, oscillation frequency zależy od prędkości przejścia i od długości wnęki. v/T przyśpieszenie ;-)

#Możemy uznać czas przed t_NL_useful użyteczny dla liniowego algorytmu?



In [ ]:
# 2.16 tau
cavity.N_eff()*2*cavity.T

In [ ]:
# 2.17 tau
cavity.T*2/np.abs(np.log(cavity.r_a*cavity.r_b))

In [ ]:
oscillation_freq[1]

In [ ]:
a = op.Cavity(t_a=0.1, r_a=0.999, r_b=0.999, L=3000, debug=False)
print(a.Finesse())

r_a_r_b_product = ut.r_a_r_b(a.Finesse())
print("r_a and r_b: {0}".format(np.sqrt(r_a_r_b_product)))

In [ ]:
cav_tau = cavity.tau()
oscillation_freq = Omega(3*cav_tau, -1*ut.critical_velocity(cavity, ut.lambd), cavity)  # rad/s or Hz?

print("oscillation frequency: {0}".format(oscillation_freq))